In [1]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch

from datasets import Audio, load_dataset
from IPython.display import Audio as PlayAudio, display

SR = 16_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", DEVICE)

stream = load_dataset(
    "CoRal-project/coral-v3",
    "read_aloud",
    split="test",
    streaming=True,
    token=True,
).cast_column("audio", Audio(sampling_rate=SR, mono=True))

by_speaker = defaultdict(list)
seen_recordings = set()
ready_speakers = []

for row in stream:
    speaker = row["id_speaker"]
    recording = row["id_recording"]

    if speaker is None or recording is None:
        continue
    if recording in seen_recordings or len(by_speaker[speaker]) >= 2:
        continue

    wave = np.asarray(row["audio"]["array"], dtype=np.float32)
    duration = len(wave) / SR

    # Берём целые реплики удобной для прослушивания длины.
    if not 2 <= duration <= 12:
        continue

    by_speaker[speaker].append({
        "id": recording,
        "wave": wave,
    })
    seen_recordings.add(recording)

    if len(by_speaker[speaker]) == 2:
        ready_speakers.append(speaker)

    if len(ready_speakers) == 2:
        break

if len(ready_speakers) < 2:
    raise ValueError("Не удалось найти двух дикторов с двумя записями.")

speaker_a, speaker_b = ready_speakers

for name, speaker in [("A", speaker_a), ("B", speaker_b)]:
    print(f"Диктор {name}: ID {speaker}")
    for clip in by_speaker[speaker]:
        print(f"  Запись {clip['id']}: {len(clip['wave']) / SR:.1f} с")

/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cuda
Диктор A: ID spe_de430b1197cf26cb5f4011656a728ee5
  Запись rec_90dda503d68dc197f80219a7d3b129a9: 10.3 с
  Запись rec_becc6eb53b441b31116584cd5c4e1c51: 7.3 с
Диктор B: ID spe_5e319f90767d47e11731d95e314e4670
  Запись rec_d0c6e79cfae4b714c45ee78d5efcda29: 5.0 с
  Запись rec_629a460b3db4caa7d962ae1493aff95d: 11.6 с


In [2]:
turns = [
    ("A", by_speaker[speaker_a][0]),
    ("B", by_speaker[speaker_b][0]),
    ("A", by_speaker[speaker_a][1]),
    ("B", by_speaker[speaker_b][1]),
]

pause = np.zeros(round(0.4 * SR), dtype=np.float32)
parts = []
reference_rows = []
position = 0

for i, (speaker_name, clip) in enumerate(turns):
    if i > 0:
        parts.append(pause)
        position += len(pause)

    wave = clip["wave"]
    start = position / SR
    parts.append(wave)
    position += len(wave)

    reference_rows.append({
        "Реплика": i + 1,
        "Диктор": speaker_name,
        "Начало, с": start,
        "Конец, с": position / SR,
    })

dialogue = np.concatenate(parts)
reference = pd.DataFrame(reference_rows)

audio_path = Path("coral_dialogue_ABAB.wav")
sf.write(audio_path, dialogue, SR)

print(f"Длительность: {len(dialogue) / SR:.1f} с")
display(reference.round(2))
display(PlayAudio(dialogue, rate=SR))

Длительность: 35.4 с


,Реплика,Диктор,"Начало, с","Конец, с"
0,1,A,0.00,10.26
1,2,B,10.66,15.64
2,3,A,16.04,23.36
3,4,B,23.76,35.40


In [3]:
from pyannote.audio import Pipeline

diarizer = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=True,
)
diarizer.to(DEVICE)

audio_input = {
    "waveform": torch.from_numpy(dialogue).unsqueeze(0),
    "sample_rate": SR,
}

output = diarizer(audio_input, num_speakers=2)

/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/pyannote/audio/core/io.py:48: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

libnvrtc.so.13: cannot open shared object file: No such file or directory
  warnings.warn(
/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:64: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends 

In [4]:
person_names = {}
predicted_rows = []

for segment, speaker_label in output.speaker_diarization:
    if speaker_label not in person_names:
        person_names[speaker_label] = f"Person {len(person_names) + 1}"

    predicted_rows.append({
        "Начало, с": segment.start,
        "Конец, с": segment.end,
        "Кто говорит": person_names[speaker_label],
    })

predicted = pd.DataFrame(
    predicted_rows,
    columns=["Начало, с", "Конец, с", "Кто говорит"],
)

display(predicted.round(2))

,"Начало, с","Конец, с",Кто говорит
0,1.50,4.69,Person 1
1,4.98,5.03,Person 1
2,5.33,9.26,Person 1
3,9.26,9.30,Person 2
4,11.27,11.98,Person 1
5,11.98,12.03,Person 2
6,12.03,12.97,Person 1
7,12.97,13.02,Person 2
8,13.02,13.26,Person 1
9,14.29,14.97,Person 1


In [5]:
comparison_rows = []

for _, turn in reference.iterrows():
    time_by_person = defaultdict(float)

    for _, segment in predicted.iterrows():
        overlap = (
            min(turn["Конец, с"], segment["Конец, с"])
            - max(turn["Начало, с"], segment["Начало, с"])
        )
        if overlap > 0:
            time_by_person[segment["Кто говорит"]] += overlap

    winner = (
        max(time_by_person, key=time_by_person.get)
        if time_by_person else "Речь не обнаружена"
    )

    comparison_rows.append({
        "Реплика": turn["Реплика"],
        "Исходный диктор": turn["Диктор"],
        "Голос модели": winner,
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

,Реплика,Исходный диктор,Голос модели
0,1,A,Person 1
1,2,B,Person 1
2,3,A,Person 1
3,4,B,Person 2


In [6]:
import importlib.util
import soundfile as sf

sf.write("coral_dialogue_ABAB.wav", dialogue, SR)
reference.to_csv("coral_dialogue_reference.csv", index=False)

print("Аудио и границы реплик сохранены.")
print("NeMo установлен:", importlib.util.find_spec("nemo") is not None)

Аудио и границы реплик сохранены.
NeMo установлен: True


In [7]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import display
from nemo.collections.asr.models import SortformerEncLabelModel

audio_path = Path("coral_dialogue_ABAB.wav")
reference = pd.read_csv("coral_dialogue_reference.csv")

wave, sr = sf.read(audio_path, dtype="float32")

if wave.ndim != 1 or sr != 16_000:
    raise ValueError(f"Ожидается моно WAV, 16000 Гц. Получено: {wave.shape}, {sr} Гц")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sortformer = SortformerEncLabelModel.from_pretrained(
    "nvidia/diar_streaming_sortformer_4spk-v2.1"
).to(device).eval()

cfg = sortformer.sortformer_modules
cfg.chunk_len = 6
cfg.chunk_right_context = 7
cfg.fifo_len = 188
cfg.spkcache_update_period = 144
cfg.spkcache_len = 188
cfg._check_streaming_parameters()

print("Устройство:", device)
print(f"Длительность записи: {len(wave) / sr:.2f} с")

[NeMo W 2026-11-11 17:34:26 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    num_spks: 4
    session_len_sec: 90
    soft_label_thres: 0.5
    soft_targets: false
    labels: null
    batch_size: 4
    shuffle: true
    num_workers: 18
    validation_mode: false
    use_lhotse: false
    use_bucketing: false
    pin_memory: true
    window_stride: 0.01
    subsampling_factor: 8
    
[NeMo W 2026-11-11 17:34:26 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    is_tarred: false
    tarred_audio_filepaths: null
    sample_rate: 16000
    num_spks: 4
    sess

[NeMo I 2026-11-11 17:34:27 save_restore_connector:285] Model SortformerEncLabelModel was successfully restored from /home/mlguest/.cache/huggingface/hub/models--nvidia--diar_streaming_sortformer_4spk-v2.1/snapshots/cd03eee90fbec18297ac31b8c21546e596b7f71c/diar_streaming_sortformer_4spk-v2.1.nemo.
Устройство: cuda
Длительность записи: 35.40 с


In [8]:
UPDATE_SEC = 2.0
FRAME_SEC = 0.08
THRESHOLD = 0.5

step_samples = round(UPDATE_SEC * sr)
duration = len(wave) / sr

frame_rows = []
next_frame = 0
previous_state = None
started = time.monotonic()

with torch.inference_mode():
    for left in range(0, len(wave), step_samples):
        right = min(left + step_samples, len(wave))
        available_sec = right / sr

        # Имитируем время поступления звука с микрофона.
        wait = available_sec - (time.monotonic() - started)
        if wait > 0:
            time.sleep(wait)

        prefix = np.ascontiguousarray(wave[:right])

        _, outputs = sortformer.diarize(
            audio=[prefix],
            sample_rate=sr,
            batch_size=1,
            include_tensor_outputs=True,
        )

        probs = torch.as_tensor(outputs[0]).detach().cpu().reshape(-1, 4)

        # Ждём правый контекст перед публикацией метки у границы.
        # В конце файла публикуем оставшиеся кадры.
        is_last = right == len(wave)
        ready_sec = available_sec if is_last else max(
            0.0, available_sec - cfg.chunk_right_context * FRAME_SEC
        )
        ready_frames = (
            int(np.ceil(ready_sec / FRAME_SEC))
            if is_last else int(np.floor(ready_sec / FRAME_SEC))
        )
        stop_frame = min(len(probs), ready_frames)

        for i in range(next_frame, stop_frame):
            t0 = i * FRAME_SEC
            t1 = min((i + 1) * FRAME_SEC, duration)
            if t1 <= t0:
                continue

            active = tuple(
                f"Person {j + 1}"
                for j in range(4)
                if probs[i, j].item() >= THRESHOLD
            )

            if active != previous_state:
                state = " + ".join(active) if active else "Пауза"
                print(
                    f"Получено {available_sec:5.2f} с | "
                    f"с {t0:5.2f} с: {state}",
                    flush=True,
                )
                previous_state = active

            for person in active:
                frame_rows.append({
                    "Начало, с": t0,
                    "Конец, с": t1,
                    "Голос": person,
                })

        next_frame = max(next_frame, stop_frame)

sort_frames = pd.DataFrame(
    frame_rows, columns=["Начало, с", "Конец, с", "Голос"]
)

[NeMo I 2026-11-11 17:34:29 vad_utils:81] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


Diarizing: 100%|██████████| 1/1 [00:00<00:00,  5.97it/s]

Получено  2.00 с | с  0.00 с: Пауза


[NeMo I 2026-11-11 17:34:31 vad_utils:81] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.


Diarizing: 100%|██████████| 1/1 [00:00<00:00,  6.75it/s]

Получено  4.00 с | с  1.60 с: Person 1


... (39 outputs left out)

Diarizing: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]

Получено 34.00 с | с 31.84 с: Person 2
[NeMo I 2026-11-11 17:35:03 vad_utils:81] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.



Diarizing: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]

Получено 35.40 с | с 34.40 с: Пауза


In [9]:
rows = []

for _, turn in reference.iterrows():
    start = turn["Начало, с"]
    end = turn["Конец, с"]

    overlap = (
        sort_frames["Конец, с"].clip(upper=end)
        - sort_frames["Начало, с"].clip(lower=start)
    ).clip(lower=0)

    totals = overlap.groupby(sort_frames["Голос"]).sum()
    winner = totals.idxmax() if not totals.empty and totals.max() > 0 else "Пауза"

    row = {
        "Реплика": int(turn["Реплика"]),
        "Исходный диктор": turn["Диктор"],
        "Sortformer": winner,
    }
    for person, seconds in totals.items():
        row[f"{person}, с"] = round(seconds, 2)

    rows.append(row)

sort_comparison = pd.DataFrame(rows)
display(sort_comparison)

,Реплика,Исходный диктор,Sortformer,"Person 1, с","Person 2, с"
0,1,A,Person 1,7.12,0.00
1,2,B,Person 2,0.00,2.80
2,3,A,Person 1,5.76,0.00
3,4,B,Person 2,0.00,7.84


Разделение 

In [17]:
from pathlib import Path
import subprocess

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

original = Path("razgovor.m4a")
fragment = Path("razgovor_overlap_16k.wav")

START = 0       # замени, если оба голоса слышны позже
DURATION = 12

subprocess.run(
    [
        ffmpeg, "-v", "error", "-y",
        "-i", str(original),
        "-ss", str(START), "-t", str(DURATION),
        "-map", "0:a:0", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(fragment),
    ],
    check=True,
)

mix, sr = sf.read(fragment, dtype="float32")
print(f"Исходный фрагмент: {START:.1f}–{START + len(mix) / sr:.1f} с")
display(Audio(mix, rate=sr, normalize=False))

Исходный фрагмент: 0.0–12.0 с


In [18]:
from clearvoice import ClearVoice

moss = ClearVoice(
    task="speech_separation",
    model_names=["MossFormer2_SS_16K"],
)

out_dir = Path("mossformer_results")
out_dir.mkdir(exist_ok=True)

moss(
    input_path=str(fragment),
    online_write=True,
    output_path=str(out_dir),
)

files = sorted(out_dir.rglob("*.wav"))
print("Полученные файлы:", [str(p) for p in files])

tracks = []
for path in files:
    audio, rate = sf.read(path, dtype="float32")
    tracks.append((path, audio, rate))

if tracks:
    peak = max(float(np.max(np.abs(audio))) for _, audio, _ in tracks)
    scale = max(1.0, peak)

    for path, audio, rate in tracks:
        print(path.name)
        display(Audio(audio.T / scale, rate=rate, normalize=False))

downloading checkpoint for MossFormer2_SS_16K


Fetching 4 files: 100%|██████████| 4/4 [00:58<00:00, 14.57s/it]


Running MossFormer2_SS_16K ...


100%|██████████| 1/1 [00:01<00:00,  1.54s/it]

Полученные файлы: ['mossformer_results/MossFormer2_SS_16K/razgovor_overlap_16k_s1.wav', 'mossformer_results/MossFormer2_SS_16K/razgovor_overlap_16k_s2.wav']
razgovor_overlap_16k_s1.wav


razgovor_overlap_16k_s2.wav


Распознование человека по векторам

In [19]:
from pathlib import Path

VOICE_DIR = Path("voices")
enroll_dir = VOICE_DIR / "enrollment"
extensions = {".wav", ".m4a", ".mp3", ".flac", ".ogg", ".aac", ".mp4"}

if not enroll_dir.is_dir():
    raise FileNotFoundError(
        f"Не найдена папка {enroll_dir.resolve()}. "
        f"Текущая папка ноутбука: {Path.cwd()}"
    )

for name in ["Эксайл", "Паша", "Булкин"]:
    folder = enroll_dir / name
    files = sorted(
        p.name for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )
    print(f"{name}: {len(files)} — {', '.join(files)}")

Эксайл: 2 — запись1.m4a, запись2.m4a
Паша: 2 — запись1.m4a, запись2.m4a
Булкин: 2 — запись1.m4a, запись2.m4a


In [20]:
from pathlib import Path
import shutil
import subprocess

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from speechbrain.inference.speaker import EncoderClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VOICE_DIR = Path("voices")
AUDIO_EXTENSIONS = {".wav", ".m4a", ".mp3", ".flac", ".ogg", ".aac", ".mp4"}

ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    import imageio_ffmpeg
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/ecapa_voxceleb",
    run_opts={"device": DEVICE},
)

print("Устройство:", DEVICE)

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Устройство: cuda


In [21]:
@torch.inference_mode()
def voice_vector(path):
    result = subprocess.run(
        [
            ffmpeg, "-v", "error",
            "-i", str(path),
            "-map", "0:a:0",
            "-ac", "1",
            "-ar", "16000",
            "-f", "f32le",
            "pipe:1",
        ],
        capture_output=True,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode(errors="replace"))

    wave = np.frombuffer(result.stdout, dtype="<f4").copy()

    if wave.size == 0 or np.max(np.abs(wave)) < 1e-7:
        raise ValueError(f"Пустая запись или тишина: {path}")

    signal = torch.from_numpy(wave).unsqueeze(0).to(DEVICE)
    vector = encoder.encode_batch(signal).reshape(-1).cpu()

    return F.normalize(vector, dim=0)

In [23]:
enroll_dir = VOICE_DIR / "enrollment"
profiles = {}

for name in ["Эксайл", "Паша", "Булкин"]:
    folder = enroll_dir / name
    files = sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
    )

    if not files:
        raise ValueError(f"В папке {folder} нет аудиозаписей.")

    vectors = torch.stack([voice_vector(path) for path in files])
    profiles[folder.name] = F.normalize(vectors.mean(dim=0), dim=0)

    print(
        f"{folder.name}: записей — {len(files)}, "
        f"чисел в векторе — {profiles[folder.name].numel()}"
    )

if len(profiles) != 3:
    raise ValueError(
        f"Найдено дикторов: {len(profiles)}. Проверь три папки в {enroll_dir}."
    )

torch.save(profiles, "voice_profiles.pt")
print("Эталоны сохранены в voice_profiles.pt")

Эксайл: записей — 2, чисел в векторе — 192
Паша: записей — 2, чисел в векторе — 192
Булкин: записей — 2, чисел в векторе — 192
Эталоны сохранены в voice_profiles.pt


In [24]:
def identify_speaker(path):
    vector = voice_vector(path)

    scores = {
        name: torch.dot(vector, profile).item()
        for name, profile in profiles.items()
    }

    ranking = sorted(scores, key=scores.get, reverse=True)
    first, second = ranking[:2]

    return {
        "Файл": Path(path).name,
        "Кто говорит": first,
        "Отрыв от 2-го места": scores[first] - scores[second],
        **{f"Сходство: {name}": value for name, value in scores.items()},
    }


test_dir = VOICE_DIR / "test"
test_files = sorted(
    p for p in test_dir.iterdir()
    if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
)

if not test_files:
    raise ValueError(f"Добавь новые записи в {test_dir}")

results = pd.DataFrame([identify_speaker(path) for path in test_files])
display(results.round(3))

,Файл,Кто говорит,Отрыв от 2-го места,Сходство: Эксайл,Сходство: Паша,Сходство: Булкин
0,пример1.m4a,Паша,0.435,0.342,0.801,0.366
1,пример2.m4a,Эксайл,0.493,0.825,0.332,0.307
2,пример3.m4a,Булкин,0.358,0.285,0.373,0.731


In [26]:
identify_speaker("voices/test/пример4.m4a")

{'Файл': 'пример4.m4a',
 'Кто говорит': 'Булкин',
 'Отрыв от 2-го места': 0.21728253364562988,
 'Сходство: Эксайл': 0.43967318534851074,
 'Сходство: Паша': 0.3724639117717743,
 'Сходство: Булкин': 0.6569557189941406}